In [3]:
import csv
from collections import defaultdict
import itertools

CSV_FILE = "student_data.csv"

first_day_mood_counts = defaultdict(int)
transition_counts = defaultdict(lambda: defaultdict(int))
mood_counts = defaultdict(int)
emission_counts = defaultdict(lambda: defaultdict(int))

students_first_day_seen = set()

# read csv
with open(CSV_FILE, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)

    rows_by_student = defaultdict(list)
    for r in reader:
        sid = r['StudentID']
        day = int(r['Day'])
        mood = r['Mood'].strip().upper()
        color = r['ShirtColor'].strip().upper()
        rows_by_student[sid].append((day, mood, color))

for sid, recs in rows_by_student.items():
    recs_sorted = sorted(recs, key=lambda x: x[0])
    if not recs_sorted:
        continue

    first_mood = recs_sorted[0][1]
    first_day_mood_counts[first_mood] += 1

    prev_mood = None
    for day, mood, color in recs_sorted:

        emission_counts[mood][color] += 1
        mood_counts[mood] += 1

        if prev_mood is not None:
            transition_counts[prev_mood][mood] += 1
        prev_mood = mood

total_students = len(rows_by_student)

# 1) Initial probabilities
initial_prob = {}
for m, c in first_day_mood_counts.items():
    initial_prob[m] = c / total_students

# 2) Transition probabilities P(next | prev)
transition_prob = {}
for prev, nxts in transition_counts.items():
    total_prev = sum(nxts.values())
    transition_prob[prev] = {}
    for nxt, cnt in nxts.items():
        transition_prob[prev][nxt] = cnt / total_prev

# 3) Emission probabilities P(color | mood)
emission_prob = {}
for m, cols in emission_counts.items():
    total_m = mood_counts[m]
    emission_prob[m] = {}
    for c, cnt in cols.items():
        emission_prob[m][c] = cnt / total_m

print("Initial probabilities P(M1):")
for m,p in initial_prob.items():
    print(f"  P({m}) = {p:.4f}")
print()

print("Transition probabilities P(M_t | M_{t-1}):")
for prev, nxts in transition_prob.items():
    for nxt, p in nxts.items():
        print(f"  P({nxt}|{prev}) = {p:.4f}")
print()

print("Emission probabilities P(color | mood):")
for m, cols in emission_prob.items():
    for c, p in cols.items():
        print(f"  P({c}|{m}) = {p:.4f}")
print()

# === Part 2: Testing Model
obs = ['R','B','G']
states = ['H','S']
results = {}

# enumerate all 8 sequences
for seq in itertools.product(states, repeat=3):
    m1, m2, m3 = seq

    p = 1.0
    # P(M1)
    p *= initial_prob.get(m1, 0.0)
    # P(R|M1)
    p *= emission_prob.get(m1, {}).get(obs[0], 0.0)
    # P(M2|M1)
    p *= transition_prob.get(m1, {}).get(m2, 0.0)
    # P(B|M2)
    p *= emission_prob.get(m2, {}).get(obs[1], 0.0)
    # P(M3|M2)
    p *= transition_prob.get(m2, {}).get(m3, 0.0)
    # P(G|M3)
    p *= emission_prob.get(m3, {}).get(obs[2], 0.0)

    results[seq] = p


print("Joint probabilities P(M1,M2,M3, R,B,G):")
for seq, p in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"  {seq} -> {p:.8e}")
# find argmax
best_seq = max(results, key=results.get)
print("\nMost likely sequence:", best_seq, "with probability", results[best_seq])


Initial probabilities P(M1):
  P(H) = 0.6000
  P(S) = 0.4000

Transition probabilities P(M_t | M_{t-1}):
  P(H|H) = 0.6545
  P(S|H) = 0.3455
  P(S|S) = 0.5500
  P(H|S) = 0.4500

Emission probabilities P(color | mood):
  P(R|H) = 0.7193
  P(G|H) = 0.2807
  P(B|S) = 0.8605
  P(G|S) = 0.1395

Joint probabilities P(M1,M2,M3, R,B,G):
  ('H', 'S', 'H') -> 1.62047402e-02
  ('H', 'S', 'S') -> 9.84532180e-03
  ('H', 'H', 'H') -> 0.00000000e+00
  ('H', 'H', 'S') -> 0.00000000e+00
  ('S', 'H', 'H') -> 0.00000000e+00
  ('S', 'H', 'S') -> 0.00000000e+00
  ('S', 'S', 'H') -> 0.00000000e+00
  ('S', 'S', 'S') -> 0.00000000e+00

Most likely sequence: ('H', 'S', 'H') with probability 0.01620474018026038
